# DivergenceLens Quickstart

This notebook demonstrates the full DivergenceLens pipeline:
1. Build a synthetic clean run and an injected (divergent) run
2. Audit both with the SDK
3. Inspect the divergence findings
4. View the provenance graph
5. Run a mini-benchmark

In [ ]:
import sys
sys.path.insert(0, '..')  # if running from examples/

from divergencelens import DivergenceLens, DivergenceLensConfig
from divergencelens.core.config import DetectionConfig

print('DivergenceLens ready')

## 1. Build a synthetic run

In [ ]:
from bench.corpus.synthetic_runs import build_clean_run

clean_run = build_clean_run('Process the dataset and write a summary report', seed=42)
print(f'Run ID: {clean_run.run_id}')
print(f'Events: {len(clean_run.events)}')
print(f'Todos: {len(clean_run.stated_artifacts.todo_transitions)}')
print(f'Tool calls: {len(clean_run.enacted_artifacts.tool_calls)}')

## 2. Audit the clean run (should be clean)

In [ ]:
config = DivergenceLensConfig(detection=DetectionConfig(enable_judge=False))
lens = DivergenceLens(config)

clean_result = lens.audit_run(clean_run)
print(f'Clean run — divergences: {len(clean_result.divergences)}')
print(f'Cells scored: {len(clean_result.cells)}')
print(f'Summary: {clean_result.summary}')

## 3. Inject a fault and audit

In [ ]:
from bench.inject.injectors import PhantomCompletionInjector, SilentFailureMaskingInjector

# Inject phantom completion: todo marked done, but we delete the supporting tool calls
injector = PhantomCompletionInjector()
injection = injector.inject(clean_run)

print(f'Injected: {injection.category.value} at step {injection.gold_step_index}')
print(f'Todo ID: {injection.gold_todo_id}')

# Audit the injected run
injected_result = lens.audit_run(injection.run)
print(f'\nDivergences found: {len(injected_result.divergences)}')
for d in injected_result.divergences:
    print(f'  [{d.severity.value}] {d.category.value} (confidence={d.confidence:.2f}): {d.rationale[:80]}')

## 4. Inspect the provenance graph

In [ ]:
from divergencelens.provenance.graph_builder import ProvenanceGraph

graph = ProvenanceGraph(clean_run)

print(f'Nodes: {graph.graph.number_of_nodes()}')
print(f'Edges: {graph.graph.number_of_edges()}')
print(f'Todo windows: {list(graph._todo_windows.keys())}')

# List node kinds
from collections import Counter
kinds = Counter(data.get('kind', 'unknown') for _, data in graph.graph.nodes(data=True))
print(f'Node kinds: {dict(kinds)}')

In [ ]:
# Visualize the graph (optional — requires matplotlib)
try:
    import matplotlib.pyplot as plt
    import networkx as nx
    
    G = graph.graph
    pos = nx.spring_layout(G, seed=42)
    node_colors = [
        '#2196F3' if G.nodes[n].get('kind') == 'tool_call' else
        '#4CAF50' if G.nodes[n].get('kind') == 'todo' else
        '#FF9800' if G.nodes[n].get('kind') == 'file_write' else
        '#9C27B0' if G.nodes[n].get('kind') == 'claim' else
        '#aaaaaa'
        for n in G.nodes()
    ]
    
    fig, ax = plt.subplots(figsize=(12, 8))
    nx.draw(G, pos, ax=ax, node_color=node_colors, node_size=300,
            with_labels=False, arrows=True, arrowsize=10,
            edge_color='#cccccc')
    
    # Legend
    from matplotlib.patches import Patch
    legend = [
        Patch(facecolor='#2196F3', label='Tool call'),
        Patch(facecolor='#4CAF50', label='Todo'),
        Patch(facecolor='#FF9800', label='File write'),
        Patch(facecolor='#9C27B0', label='Claim'),
        Patch(facecolor='#aaaaaa', label='Other'),
    ]
    ax.legend(handles=legend, loc='upper left')
    ax.set_title('Provenance Graph — Clean Run')
    plt.tight_layout()
    plt.show()
except ImportError:
    print('matplotlib not available for visualization')

## 5. Mini-benchmark: all divergence categories

In [ ]:
from bench.inject.injectors import get_all_injectors

injectors = get_all_injectors()
print(f'Testing {len(injectors)} injectors on the same base run:\n')

for inj in injectors:
    result = inj.inject(clean_run)
    if result is None:
        print(f'  {inj.name}: not applicable to this run')
        continue
    
    audit = lens.audit_run(result.run)
    detected = any(d.category == inj.category for d in audit.divergences)
    status = '✅ DETECTED' if detected else '❌ MISSED'
    conf = max((d.confidence for d in audit.divergences if d.category == inj.category), default=0.0)
    print(f'  {inj.name}: {status} (confidence={conf:.2f})')

## 6. Full benchmark run (3 seeds)

In [ ]:
from bench.metrics.compute import run_benchmark
import json

results = run_benchmark(split='test', n_seeds=3, enable_judge=False, output_dir='../results/')
print(json.dumps({
    'mean_f1': results['mean_f1'],
    'std_f1': results['std_f1'],
    'ci_95': results['ci_95'],
}, indent=2))

## 7. SDK usage with LangSmith (requires API key)

```python
import os
os.environ['LANGSMITH_API_KEY'] = 'your-key'

lens = DivergenceLens()
result = lens.audit_langsmith_run('your-run-id')

for div in result.divergences:
    print(f'[{div.severity.value}] {div.category.value}: {div.rationale}')
```

## 8. Middleware (real-time auditing)

```python
from deepagents import create_deep_agent
from divergencelens.runtime.middleware import DivergenceMiddleware

agent = create_deep_agent(
    model='anthropic:claude-sonnet-4-6',
    middleware=[DivergenceMiddleware()],
    tools=[...],
)

result = await agent.ainvoke({'messages': [HumanMessage('Process the data')]})
```